In [1]:
import pandas as pd
import numpy as np
import joblib
from itertools import combinations

In [2]:
outcome_model = joblib.load('../src/outcome_model.pkl')
home_goals_model = joblib.load('../src/home_goals_model.pkl')
away_goals_model = joblib.load('../src/away_goals_model.pkl')

team_stats = pd.read_csv('../data/team_stats.csv')
print(team_stats.head(5))
print(len(team_stats))

            team  total_games  avg_goal_scored  avg_goal_conceded  win_rate  \
0         Canada           41            1.184              1.158     0.341   
1         Mexico           58            1.304              1.268     0.345   
2  United States           55            1.377              1.132     0.418   
3      Australia           31            1.207              1.103     0.419   
4           Iraq           29            0.577              1.308     0.207   

   draw_rate  home_goals_scored  home_goals_conceded  home_win_rate  \
0      0.244              1.467                0.800          0.389   
1      0.293              1.324                1.351          0.256   
2      0.218              1.628                1.116          0.489   
3      0.161              1.357                0.714          0.571   
4      0.207              0.750                1.417          0.231   

   away_goals_scored  away_goals_conceded  away_win_rate  fifa_rank  \
0              1.000       

In [10]:
def predict_match(home_team, away_team):
    """
    Predict match using trained XGBoost model.
    """
    home_row = team_stats[team_stats['team']==home_team]
    away_row = team_stats[team_stats['team']==away_team]

    if home_row.empty or away_row.empty:
        return None
        
    
    home = home_row.iloc[0]
    away = away_row.iloc[0]

    features = pd.DataFrame([{
        'home_goals_scored_avg':home['avg_goal_scored'],
        'home_goals_conceded_avg': home['avg_goal_conceded'],
        'home_win_rate': home['win_rate'],
        'home_draw_rate': home['draw_rate'],
        'home_form_points': home.get('form_points', 1.0),
        'home_rank': home['fifa_rank'],
        'away_goals_scored_avg': away['avg_goal_scored'],
        'away_goals_conceded_avg': away['avg_goal_conceded'],
        'away_win_rate': away['win_rate'],
        'away_draw_rate': away['draw_rate'],
        'away_form_points': away.get('form_points', 1.0),
        'away_rank': away['fifa_rank'],
        'rank_diff': away['fifa_rank'] - home['fifa_rank'],
        'goal_avg_diff': home['avg_goal_scored'] - away['avg_goal_scored'],
        'form_diff': home.get('form_points', 1.0) - away.get('form_points', 1.0),
    }])

    probs = outcome_model.predict_proba(features)[0]
    away_win_p = round(probs[0] * 100, 1)
    draw_p = round(probs[1] * 100, 1)
    home_win_p = round(probs[2] * 100, 1)

    home_goals = int(max(0, round(home_goals_model.predict(features)[0])))
    away_goals = int(max(0, round(away_goals_model.predict(features)[0])))

    if home_win_p > away_win_p and home_win_p > draw_p:
        winner = 'home'
    elif away_win_p > home_win_p and away_win_p > draw_p:
        winner = 'away'
    else:
        winner = 'draw'
    
    return{
        'home_team': home_team,
        'away_team': away_team,
        'predicted_home_goals': home_goals,
        'predicted_away_goals': away_goals,
        'home_win_prob': round(home_win_p, 2),
        'draw_prob': round(draw_p,2),
        'away_win_prob': round(away_win_p, 2),
        'predicted_winner': winner
    }

In [16]:
def predict_extras(home_team, away_team):
    home_row = team_stats[team_stats['team'] == home_team]
    away_row = team_stats[team_stats['team'] == away_team]

    if home_row.empty or away_row.empty:
        return None

    home = home_row.iloc[0]
    away = away_row.iloc[0]

    rank_factor = (home['fifa_rank'] + away['fifa_rank']) / 200

    return {
        'corners': int(round(10.2 + rank_factor * 2, 0)),
        'yellow_cards': int(round(3.1 + rank_factor * 0.5, 0)),
        'red_cards': 0
    }

In [17]:
#testing
test = predict_match("Brazil", 'France')
if test is None:
    print("Cannot Predict: team data is missing.")
else:
    print("Brazil vs France")
    print(f" Score: {test['predicted_home_goals']} - {test['predicted_away_goals']}")
    print(f'Home Win: {test['home_win_prob']:.1f}% Draw: {test['draw_prob']:.1f}% Away Win: {test['away_win_prob']:.1f}%')
    print(f"Winner: {test['predicted_winner']}")



Brazil vs France
 Score: 1 - 1
Home Win: 42.4% Draw: 27.4% Away Win: 30.2%
Winner: home


In [18]:
groups = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czechia'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['United States', 'Paraguay', 'Australia', 'Türkiye'],
    'E': ['Germany', 'Curaçao', "Côte d'Ivoire"],
    'F': ['Netherlands','Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain','Cabo Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'Democratic Republic of Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

In [14]:
def simulate_group(group_name, teams):
    matches = []
    for home_team, away_team in combinations(teams, 2):
        pred   = predict_match(home_team, away_team)
        extras = predict_extras(home_team, away_team)
        if pred is None or extras is None:
            print("Cannot predict, team data is missing.")
            return None
        
        matches.append({
            'group':      group_name,
            'home_team':  home_team,
            'away_team':  away_team,
            'home_goals': pred['predicted_home_goals'],
            'away_goals': pred['predicted_away_goals'],
            'winner':     pred['predicted_winner'],
            'home_win_prob': pred['home_win_prob'],
            'draw_prob':     pred['draw_prob'],
            'away_win_prob': pred['away_win_prob'],
            'corners':       extras['corners'],
            'yellow_cards':  extras['yellow_cards'],
            'red_cards':     extras['red_cards'],
        })
    return matches

def calculate_standings(group_name, teams, matches):
    standings = {team: {
        'team': team, 'group': group_name,
        'played': 0, 'won': 0, 'drawn': 0, 'lost': 0,
        'goals_for': 0, 'goals_against': 0,
        'goal_diff': 0, 'points': 0
    } for team in teams}
    
    for m in matches:
        home = m['home_team']
        away = m['away_team']
        hg   = m['home_goals']
        ag   = m['away_goals']
        
        standings[home]['played'] += 1
        standings[away]['played'] += 1
        standings[home]['goals_for']     += hg
        standings[home]['goals_against'] += ag
        standings[away]['goals_for']     += ag
        standings[away]['goals_against'] += hg
        
        if hg > ag:
            standings[home]['won']    += 1
            standings[home]['points'] += 3
            standings[away]['lost']   += 1
        elif ag > hg:
            standings[away]['won']    += 1
            standings[away]['points'] += 3
            standings[home]['lost']   += 1
        else:
            standings[home]['drawn']  += 1
            standings[home]['points'] += 1
            standings[away]['drawn']  += 1
            standings[away]['points'] += 1
    
    for team in standings:
        standings[team]['goal_diff'] = (
            standings[team]['goals_for'] - standings[team]['goals_against']
        )
    
    table = pd.DataFrame(standings.values())
    table = table.sort_values(
        ['points', 'goal_diff', 'goals_for'],
        ascending=[False, False, False]
    ).reset_index(drop=True)
    table.index += 1
    return table

In [20]:
all_matches   = []
all_standings = []
group_winners = {}
group_runners = {}
all_thirds    = []

total_expected = 0
for group_name, teams in groups.items():
    matches = len(list(combinations(teams, 2)))
    total_expected +=  matches
    print(f'Group {group_name}: {matches} matches')

print(f'Expected total group matches: {total_expected}')

print("Simulating group stage...\n")

for group_name, teams in groups.items():
    g_matches = simulate_group(group_name, teams)

    if g_matches is None:
        print(f"Skipping Group {group_name}, missing team data.")
    all_matches.extend(g_matches)
    
    table = calculate_standings(group_name, teams, g_matches)
    all_standings.append(table)
    
    group_winners[group_name] = table.iloc[0]['team']
    group_runners[group_name] = table.iloc[1]['team']
    third = table.iloc[2]
    
    all_thirds.append({
        'team':      third['team'],
        'group':     group_name,
        'points':    third['points'],
        'goal_diff': third['goal_diff'],
        'goals_for': third['goals_for']
    })
    
    print(f"Group {group_name}: {group_winners[group_name]:<22} {group_runners[group_name]}")

# Best 8 third place teams
thirds_df  = pd.DataFrame(all_thirds).sort_values(
    ['points', 'goal_diff', 'goals_for'],
    ascending=[False, False, False]
).reset_index(drop=True)
best_thirds = thirds_df.head(8)['team'].tolist()

print(f"\nBest 8 third-place teams:")
for t in best_thirds:
    print(f"  → {t}")

print(f"\nTotal group matches: {len(all_matches)}")

Group A: 6 matches
Group B: 6 matches
Group C: 6 matches
Group D: 6 matches
Group E: 3 matches
Group F: 6 matches
Group G: 6 matches
Group H: 6 matches
Group I: 6 matches
Group J: 6 matches
Group K: 6 matches
Group L: 6 matches
Expected total group matches: 69
Simulating group stage...

Group A: Mexico                 South Africa
Group B: Switzerland            Canada
Group C: Brazil                 Morocco
Group D: Australia              United States
Group E: Germany                Curaçao
Group F: Netherlands            Japan
Group G: Belgium                Egypt
Group H: Spain                  Uruguay
Group I: France                 Senegal
Group J: Argentina              Algeria
Group K: Portugal               Colombia
Group L: Croatia                England

Best 8 third-place teams:
  → Tunisia
  → Austria
  → Iran
  → Czechia
  → Scotland
  → Paraguay
  → Qatar
  → Norway

Total group matches: 69


In [24]:
def simulate_knockout_round(matchups, round_name, multiplier):
    results = []
    winners = []
    losers  = []
    
    print(f"\n{'='*55}")
    print(f"  {round_name.upper()}  (×{multiplier})")
    print(f"{'='*55}")
    
    for home, away in matchups:
        pred   = predict_match(home, away)
        extras = predict_extras(home, away)
        
        
        if pred['predicted_winner'] == 'draw':
            winner = home if pred['home_win_prob'] >= pred['away_win_prob'] else away
            loser  = away if winner == home else home
            goes_to_pens = True
        else:
            winner = home if pred['predicted_winner'] == 'home' else away
            loser  = away if winner == home else home
            goes_to_pens = pred['home_win_prob'] < 40 or pred['away_win_prob'] < 40
        
        winners.append(winner)
        losers.append(loser)
        
        score = f"{pred['predicted_home_goals']}-{pred['predicted_away_goals']}"
        pens  = " - pens" if goes_to_pens else ""
        print(f"  {home:<22} vs {away:<22} → {score}{pens}   {winner}")
        
        results.append({
            'round':       round_name,
            'multiplier':  multiplier,
            'home_team':   home,
            'away_team':   away,
            'home_goals':  pred['predicted_home_goals'],
            'away_goals':  pred['predicted_away_goals'],
            'winner':      winner,
            'penalties':   goes_to_pens,
            'home_win_prob': pred['home_win_prob'],
            'away_win_prob': pred['away_win_prob'],
            'corners':       extras['corners'],
            'yellow_cards':  extras['yellow_cards'],
            'red_cards':     extras['red_cards'],
        })
    
    return results, winners, losers

In [25]:
all_knockout = []

# Round of 32
r32_matchups = [
    (group_winners['A'], group_runners['B']),
    (group_winners['C'], group_runners['D']),
    (group_winners['E'], group_runners['F']),
    (group_winners['G'], group_runners['H']),
    (group_winners['I'], group_runners['J']),
    (group_winners['K'], group_runners['L']),
    (group_winners['B'], group_runners['A']),
    (group_winners['D'], group_runners['C']),
    (group_winners['F'], group_runners['E']),
    (group_winners['H'], group_runners['G']),
    (group_winners['J'], group_runners['I']),
    (group_winners['L'], group_runners['K']),
    (group_winners['A'], best_thirds[0]),
    (group_winners['E'], best_thirds[1]),
    (group_winners['I'], best_thirds[2]),
    (group_winners['C'], best_thirds[3]),
]

r32_res, r32_winners, _ = simulate_knockout_round(r32_matchups, 'Round of 32', 1)
all_knockout.extend(r32_res)

# Round of 16
r16_matchups = [(r32_winners[i], r32_winners[i+1])
                for i in range(0, len(r32_winners), 2)]
r16_res, r16_winners, _ = simulate_knockout_round(r16_matchups, 'Round of 16', 2)
all_knockout.extend(r16_res)

# Quarter-finals
qf_matchups = [(r16_winners[i], r16_winners[i+1])
               for i in range(0, len(r16_winners), 2)]
qf_res, qf_winners, qf_losers = simulate_knockout_round(qf_matchups, 'Quarter-final', 4)
all_knockout.extend(qf_res)

# Semi-finals
sf_matchups = [(qf_winners[i], qf_winners[i+1])
               for i in range(0, len(qf_winners), 2)]
sf_res, sf_winners, sf_losers = simulate_knockout_round(sf_matchups, 'Semi-final', 8)
all_knockout.extend(sf_res)

# Third place playoff
tp_res, tp_winners, _ = simulate_knockout_round(
    [(sf_losers[0], sf_losers[1])], 'Third-place playoff', 8)
all_knockout.extend(tp_res)

# THE FINAL
final_res, final_winner, _ = simulate_knockout_round(
    [(sf_winners[0], sf_winners[1])], 'FINAL', 16)
all_knockout.extend(final_res)

print(f"\n{'-' * 20}")
print(f"  FIFA WORLD CUP 2026 WINNER: {final_winner[0].upper()}")
print(f"{'-' * 20}")


  ROUND OF 32  (×1)
  Mexico                 vs Canada                 → 2-1 - pens   Mexico
  Brazil                 vs United States          → 3-1 - pens   Brazil
  Germany                vs Japan                  → 1-2 - pens   Japan
  Belgium                vs Uruguay                → 1-2 - pens   Belgium
  France                 vs Algeria                → 2-1 - pens   France
  Portugal               vs England                → 2-2 - pens   Portugal
  Switzerland            vs South Africa           → 2-1 - pens   Switzerland
  Australia              vs Morocco                → 1-2 - pens   Morocco
  Netherlands            vs Curaçao                → 3-1 - pens   Netherlands
  Spain                  vs Egypt                  → 1-1 - pens   Spain
  Argentina              vs Senegal                → 2-2 - pens   Argentina
  Croatia                vs Colombia               → 1-2 - pens   Colombia
  Mexico                 vs Tunisia                → 1-1 - pens   Tunisia
  Germany   

In [26]:
# Save group stage
group_df = pd.DataFrame(all_matches)
group_df.to_csv('../outputs/group_stage_predictions.csv', index=False)

standings_df = pd.concat(all_standings)
standings_df.to_csv('../outputs/group_standings.csv', index=False)

# Save knockout
knockout_df = pd.DataFrame(all_knockout)
knockout_df.to_csv('../outputs/knockout_predictions.csv', index=False)

print("All ML predictions saved!")
print(f"Group matches:    {len(group_df)}")
print(f"Knockout matches: {len(knockout_df)}")
print(f"\n Predicted Champion: {final_winner[0]}")

All ML predictions saved!
Group matches:    69
Knockout matches: 32

 Predicted Champion: Brazil
